In [64]:
#!pip install --upgrade torchao

In [65]:
%reset -s -f

In [66]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from peft import get_peft_model, LoraConfig, TaskType
import wandb
from kaggle_secrets import UserSecretsClient

In [67]:
# All files under the input directory
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# SETUP & CONFIGURATION

In [68]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(seed=42)

# Global constants
TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
LABEL_MAP = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [69]:
user_secrets = UserSecretsClient()
wandb_api_key = user_secrets.get_secret("WandB-API")
wandb.login(key=wandb_api_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

# METRIC EVALUATION FUNCTION

In [70]:
def calculate_metrics(eval_preds):
    logits, labels = eval_preds
    preds = np.argsort(logits, axis=-1)[:, ::-1] # Sort in descending order

    top1_preds = preds[:, 0]
    accuracy = accuracy_score(labels, top1_preds)
    f1 = f1_score(labels, top1_preds, average="macro")

    map3_score = 0.0
    for i in range(len(labels)):
        true_label = labels[i]
        for rank in range(3):
            if preds[i, rank] == true_label:
                map3_score += 1.0 / (rank + 1)
                break
    map3_score /= len(labels)
    
    return {
        "accuracy": accuracy,
        "f1_score": f1,
        "map@3": map3_score
    }

# Model-1: TF-IDF

In [71]:
df = pd.read_csv(TRAIN_PATH)

# Step 1: Case normalization - Convert all text to lowercase
df['prompt_normalized'] = df['prompt'].apply(lambda x: x.lower())

# Display example of case normalization effect
print("\nCase Normalization Example:")
original_text = df['prompt'].iloc[0][:100]
normalized_text = df['prompt_normalized'].iloc[0][:100]
print(f"Original: {original_text}...")
print(f"Normalized: {normalized_text}...")


Case Normalization Example:
Original: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and ...
Normalized: pick the best possible answer: what is martin heidegger's view on the relationship between time and ...


In [72]:
count = (~df['prompt'].str.contains(r'[?:]', na=False)).sum()
count

np.int64(0)

In [73]:
# Step 2: Punctuation removal
from string import punctuation
print("\nPunctuation characters to remove:")
print(punctuation)

# Remove all punctuation characters from the normalized text
print("\nApplying punctuation removal...")
df['clean_text'] = df['prompt_normalized'].apply(
    lambda x: ''.join(c for c in x if c not in punctuation or c in '?:')
)

# Display example of punctuation removal effect
print("\nPunctuation Removal Example:")
normalized_with_punct = df['prompt_normalized'].iloc[0][:100]
cleaned_text = df['clean_text'].iloc[0][:100]
print(f"Before: {normalized_with_punct}...")
print(f"After: {cleaned_text}...")


Punctuation characters to remove:
!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~

Applying punctuation removal...

Punctuation Removal Example:
Before: pick the best possible answer: what is martin heidegger's view on the relationship between time and ...
After: pick the best possible answer: what is martin heideggers view on the relationship between time and h...


In [74]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id                 2000 non-null   int64 
 1   prompt             2000 non-null   object
 2   A                  2000 non-null   object
 3   B                  2000 non-null   object
 4   C                  2000 non-null   object
 5   D                  2000 non-null   object
 6   E                  2000 non-null   object
 7   answer             2000 non-null   object
 8   prompt_normalized  2000 non-null   object
 9   clean_text         2000 non-null   object
dtypes: int64(1), object(9)
memory usage: 156.4+ KB


.describe(include='all') generates descriptive statistics. Using include='all' ensures it also summarizes object/string columns showing counts, unique values, and the most frequent value.

.info() prints a concise summary of the DataFrame, including the index dtype and columns, non-null values, and memory usage.


In [75]:
df.head(5)

,id,prompt,A,B,C,D,E,answer,prompt_normalized,clean_text
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,pick the best possible answer: what is martin ...,pick the best possible answer: what is martin ...
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,what is accelerator-based light-ion fusion?,what is acceleratorbased lightion fusion?
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C,determine the correct option: what is the term...,determine the correct option: what is the term...
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,select the most accurate option: what is marti...,select the most accurate option: what is marti...
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A,identify the correct statement: what is the co...,identify the correct statement: what is the co...


In [76]:
# Stratified split to ensure answer distributions match
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['answer'])
label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

In [77]:
# To build a robust vocabulary, we compile all prompts and options from the training set
train_texts = train_df['clean_text'].tolist()
for opt in ['A', 'B', 'C', 'D', 'E']:
    train_texts.extend(train_df[opt].tolist())

In [78]:
# Initialize vectorizer (removing English stop words helps focus on key terms)
vectorizer = TfidfVectorizer(stop_words='english',
                             max_df=0.95)          # Ignores words that appear in over 95% of the text (useless for discrimination))

# Fit the model to learn the vocabulary and IDF weights from the training data
vectorizer.fit(train_texts) 

TfidfVectorizer(max_df=0.95, stop_words='english')

In [79]:
# EVALUATE ON VALIDATION DATA
val_probs = []
val_labels = []

# Process row by row for validation
for idx, row in val_df.iterrows():
    prompt = str(row['prompt'])
    options = [str(row[opt]) for opt in ['A', 'B', 'C', 'D', 'E']]
    
    # Transform text into TF-IDF vectors
    prompt_vec = vectorizer.transform([prompt])
    options_vec = vectorizer.transform(options)
    
    # Calculate cosine similarity between the prompt and each of the 5 options
    sims = cosine_similarity(prompt_vec, options_vec)[0] 
    
    val_probs.append(sims)
    val_labels.append(label_map[row['answer']])

# Convert to numpy arrays for metric calculation
val_probs = np.array(val_probs)
val_labels = np.array(val_labels)

In [80]:
# Model Evaluation
# Sort predictions in descending order to rank the highest similarities first
preds_ranked = np.argsort(val_probs, axis=-1)[:, ::-1]

top1_preds = preds_ranked[:, 0]
val_accuracy = accuracy_score(val_labels, top1_preds)
val_f1 = f1_score(val_labels, top1_preds, average='macro')

# MAP@3 Metric
map3_score = 0.0
for i in range(len(val_labels)):
    true_label = val_labels[i]
    for rank in range(3):
        if preds_ranked[i, rank] == true_label:
            map3_score += 1.0 / (rank + 1)
            break
map3_score /= len(val_labels)

print("\n--- TF-IDF Validation Performance ---")
print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Validation F1 Score: {val_f1:.4f}")
print(f"Validation MAP@3:    {map3_score:.4f}")


--- TF-IDF Validation Performance ---
Validation Accuracy: 0.1150
Validation F1 Score: 0.1052
Validation MAP@3:    0.2721


## HUGGING FACE TRANSFORMER DATASETS

In [81]:
class HuggingFaceMCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256, is_test=False):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_test = is_test
        self.options = ['A', 'B', 'C', 'D', 'E']

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row['prompt'])
        
        choices_inputs = []
        for opt in self.options:
            option_text = str(row[opt])
            choices_inputs.append((prompt, option_text))
            
        # Standard tokenization structure matching [Batch_Size, Num_Choices, Max_Length]
        features = self.tokenizer(
            [text[0] for text in choices_inputs],
            [text[1] for text in choices_inputs],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        
        item = {
            "input_ids": features["input_ids"],
            "attention_mask": features["attention_mask"]
        }
        
        if not self.is_test:
            item["labels"] = torch.tensor(LABEL_MAP[row['answer']], dtype=torch.long)
            
        return item

# Data collator to enforce precise shapes for Hugging Face multi-choice pipeline execution
def mcq_data_collator(features):
    batch = {}
    batch["input_ids"] = torch.stack([f["input_ids"] for f in features])
    batch["attention_mask"] = torch.stack([f["attention_mask"] for f in features])
    if "labels" in features[0]:
        batch["labels"] = torch.stack([f["labels"] for f in features])
    return batch

# Model-2: PRETRAINED DeBERTa

In [82]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

# Stratified validation splits
train_split, val_split = train_test_split(train_df, test_size=0.2, random_state=42, stratify=train_df['answer'])

deberta_ckpt = "microsoft/deberta-v3-small"
deberta_tokenizer = AutoTokenizer.from_pretrained(deberta_ckpt)
deberta_model = AutoModelForMultipleChoice.from_pretrained(deberta_ckpt).to(DEVICE)
deberta_train_ds = HuggingFaceMCQDataset(train_split, deberta_tokenizer, max_len=128)
deberta_val_ds = HuggingFaceMCQDataset(val_split, deberta_tokenizer, max_len=128)

NUM_EPOCHS = 3 

deberta_args = TrainingArguments(
    output_dir="./deberta_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,                
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,                   
    max_grad_norm=0.5,
    per_device_train_batch_size=8,      
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,   
    num_train_epochs=NUM_EPOCHS,
    load_best_model_at_end=True,
    greater_is_better=True,
    save_total_limit=2,
    report_to="wandb",
    run_name="pretrained-deberta-run",
    logging_steps=10,
)

deberta_trainer = Trainer(
    model=deberta_model,
    args=deberta_args,
    train_dataset=deberta_train_ds,
    eval_dataset=deberta_val_ds,
    data_collator=mcq_data_collator,
    compute_metrics=calculate_metrics,
)
deberta_trainer.train()
wandb.finish()

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias                 

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1 Score,Map@3
1,8.016992,3.371094,0.112500,0.109186,0.211667
2,6.090820,2.722656,0.435000,0.431476,0.595000
3,4.639453,1.925781,0.582500,0.584371,0.711667


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

eval/accuracy,▁▆█
eval/f1_score,▁▆█
eval/loss,█▅▁
eval/map@3,▁▆█
eval/runtime,█▁▆
eval/samples_per_second,▁█▃
eval/steps_per_second,▁█▃
train/epoch,▁▂▂▃▃▃▄▄▅▅▅▅▆▇▇▇███
train/global_step,▁▁▂▃▃▃▃▄▅▅▅▅▆▇▇▇███
train/grad_norm,▁█▁▅▁▁▁▁▁▁▁▁▁▁▁
+2,...


In [83]:
# # PREDICTIONS FOR MODEL 2 (DeBERTa)
# # 1. Create the test dataset and dataloader for DeBERTa
# test_deberta_ds = HuggingFaceMCQDataset(test_df, deberta_tokenizer, max_len=128, is_test=True)
# deberta_test_loader = DataLoader(test_deberta_ds, batch_size=4, shuffle=False, collate_fn=mcq_data_collator)

# # 2. Set the model to evaluation mode
# deberta_model.eval()
# deberta_probs = []

# # 3. Run inference without calculating gradients
# with torch.no_grad():
#     for batch in deberta_test_loader:
#         inputs = {k: v.to(DEVICE) for k, v in batch.items()}
#         logits = deberta_model(**inputs).logits
#         # Convert logits to probabilities
#         probs = F.softmax(logits, dim=-1)
#         deberta_probs.append(probs.cpu().numpy())
        
# # Concatenate all batches into a single numpy array
# deberta_probs = np.concatenate(deberta_probs, axis=0)

# # 4. Extract Top-3 space-separated string maps for output submissions
# deberta_submission_predictions = []
# for probs in deberta_probs:
#     # Sort indices in descending order based on probability and grab the top 3
#     top3_indices = np.argsort(probs)[::-1][:3]
#     # Map numeric indices back to 'A', 'B', 'C', 'D', 'E'
#     top3_labels = [INV_LABEL_MAP[idx] for idx in top3_indices]
#     deberta_submission_predictions.append(" ".join(top3_labels))

# Model-3: Fine-Tuned RoBERTa + LoRA Layers

In [84]:
# TRAINING MODEL 3 (Fine-Tuned RoBERTa + LoRA Layers)
roberta_ckpt = "roberta-base"
roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_ckpt)

base_roberta_model = AutoModelForMultipleChoice.from_pretrained(roberta_ckpt)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)
roberta_peft_model = get_peft_model(base_roberta_model, lora_config).to(DEVICE)

roberta_train_ds = HuggingFaceMCQDataset(train_split, roberta_tokenizer, max_len=128)
roberta_val_ds = HuggingFaceMCQDataset(val_split, roberta_tokenizer, max_len=128)

roberta_args = TrainingArguments(
    output_dir="./roberta_lora_results",
    eval_strategy="epoch",  
    save_strategy="epoch",
    learning_rate=5e-4, 
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=5,
    report_to="wandb",
    run_name="peft-roberta-lora-run",
    logging_steps=10
)

roberta_trainer = Trainer(
    model=roberta_peft_model,
    args=roberta_args,
    train_dataset=roberta_train_ds,
    eval_dataset=roberta_val_ds,
    data_collator=mcq_data_collator,
    compute_metrics=calculate_metrics,
)

roberta_trainer.train()
wandb.finish()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForMultipleChoice LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.weight               | MISSING    | 
roberta.pooler.dense.bias       | MISSING    | 
roberta.pooler.dense.weight     | MISSING    | 
classifier.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1 Score,Map@3
1,2.798015,2.818641,0.547500,0.535739,0.685417
2,1.938804,1.482672,0.717500,0.714259,0.803333
3,1.677747,1.148803,0.777500,0.772767,0.852917
4,1.290692,0.955241,0.825000,0.822561,0.890417
5,1.356729,0.879734,0.840000,0.836593,0.897083


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

eval/accuracy,▁▅▇██
eval/f1_score,▁▅▇██
eval/loss,█▃▂▁▁
eval/map@3,▁▅▇██
eval/runtime,▇▇█▁▁
eval/samples_per_second,▂▂▁██
eval/steps_per_second,▂▂▁██
train/epoch,▁▁▁▁▂▂▂▂▂▂▂▂▂▃▃▃▄▄▄▄▄▄▄▄▅▅▆▆▆▆▇▇▇▇▇▇████
train/global_step,▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇████
train/grad_norm,▁▁▁▁▁▁▂▂▂▁▂▂▂▂▂▃▃▃▂▂▃▂▃▂▃▄▂▁▃▃▃█▃▂▂▃▃▂▄▂
+2,...


In [85]:
# Create the test dataset and dataloader for RoBERTa
test_roberta_ds = HuggingFaceMCQDataset(test_df, roberta_tokenizer, max_len=128, is_test=True)
roberta_test_loader = DataLoader(test_roberta_ds, batch_size=4, shuffle=False, collate_fn=mcq_data_collator)

# Set the model to evaluation mode
roberta_peft_model.eval()
roberta_probs = []

# Run inference without calculating gradients
with torch.no_grad():
    for batch in roberta_test_loader:
        inputs = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = roberta_peft_model(**inputs).logits
        # Convert logits to probabilities
        probs = F.softmax(logits, dim=-1)
        roberta_probs.append(probs.cpu().numpy())
        
# Concatenate all batches into a single numpy array
roberta_probs = np.concatenate(roberta_probs, axis=0)

# Extract Top-3 space-separated string maps for output submissions
roberta_submission_predictions = []
for probs in roberta_probs:
    # Sort indices in descending order based on probability and grab the top 3
    top3_indices = np.argsort(probs)[::-1][:3]
    # Map numeric indices back to 'A', 'B', 'C', 'D', 'E'
    top3_labels = [INV_LABEL_MAP[idx] for idx in top3_indices]
    roberta_submission_predictions.append(" ".join(top3_labels))
    
# Kaggle submission tracking file
submission_df = pd.DataFrame({
    "id": test_df["id"],
    "Prediction": roberta_submission_predictions
})

submission_df.to_csv("submission.csv", index=False)
print("Inference completed successfully. Output saved to: submission.csv")

Inference completed successfully. Output saved to: submission.csv
